# Final Full Fine-Tuning Baseline (mT5)

**Model:** google/mt5-small  
**Task:** Binary Sentiment Classification (Positive vs Negative)  
**Dataset:** Malayalam–English Code-Mixed (FIRE 2020 – Dravidian CodeMix)

---

## Training Configuration

- Full fine-tuning (all parameters trainable)
- Binary classification (Positive vs Negative only)
- Training split oversampled to balance classes (2018 Positive / 2018 Negative)
- Original dev and test splits preserved (no oversampling applied)
- 3 training epochs
- Learning rate: 3e-5
- fp16 disabled (for training stability)
- Trained on GPU using HuggingFace Transformers

---

## Final Test Results

- **Accuracy:** 0.846  
- **Weighted F1:** 0.811  
- **Macro F1:** 0.656  

The model predicts both sentiment classes and does not collapse to the majority class.  
Macro F1 and confusion matrix are reported to properly evaluate minority class performance.

---

## Saved Model

The trained model weights are stored in:

`mt5_finetuned_updated`

Due to GitHub file size limits, model weights are hosted separately in Google Drive.

## Setup & Environment

Install required libraries, check GPU, and mount Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

train = pd.read_csv("/content/drive/MyDrive/Project2_preprocessing_mal_en/clean_train.csv")
dev   = pd.read_csv("/content/drive/MyDrive/Project2_preprocessing_mal_en/clean_dev.csv")
test  = pd.read_csv("/content/drive/MyDrive/Project2_preprocessing_mal_en/clean_test.csv")

print("Train shape:", train.shape)
print("Dev shape:",   dev.shape)
print("Test shape:",  test.shape)

In [ ]:
# Keep only Positive and Negative
train = train[train["category"].isin(["Positive", "Negative"])]
dev   = dev[dev["category"].isin(["Positive", "Negative"])]
test  = test[test["category"].isin(["Positive", "Negative"])]

print("Filtered Train shape:", train.shape)
print("Filtered Dev shape:",   dev.shape)
print("Filtered Test shape:",  test.shape)

## Convert to mT5 Format

In [ ]:
def convert_to_finetune_format(df):
    formatted = []
    for _, row in df.iterrows():
        formatted.append({
            "input_text":  "sentiment: " + str(row["text"]),
            "target_text": row["category"].lower().strip()
        })
    return formatted

train_data = convert_to_finetune_format(train)
dev_data   = convert_to_finetune_format(dev)
test_data  = convert_to_finetune_format(test)

print(f"Prepared {len(train_data)} train, {len(dev_data)} dev, {len(test_data)} test examples")

In [ ]:
from collections import defaultdict
import random
random.seed(42)

# group examples by label
by_label = defaultdict(list)
for ex in train_data:
    by_label[ex["target_text"]].append(ex)

pos = by_label["positive"]
neg = by_label["negative"]

print("Before oversampling:", len(pos), "pos,", len(neg), "neg")

# Oversample negatives to match positives
if len(neg) < len(pos):
    neg_balanced = neg + random.choices(neg, k=(len(pos) - len(neg)))
else:
    neg_balanced = neg

train_data = pos + neg_balanced
random.shuffle(train_data)

print("After oversampling:",
      sum(1 for x in train_data if x["target_text"]=="positive"), "pos,",
      sum(1 for x in train_data if x["target_text"]=="negative"), "neg")

## Model + Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print("Model loaded:", MODEL_NAME)

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", model.device)

In [ ]:
model = model.to("cuda")
print("Device after move:", model.device)

## Tokenization

In [ ]:
def tokenize_example(example):
    model_input = tokenizer(
        example["input_text"],
        max_length=64,
        padding="max_length",
        truncation=True
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            example["target_text"],
            max_length=8,
            padding="max_length",
            truncation=True
        )

    label_ids = [
        (token if token != tokenizer.pad_token_id else -100)
        for token in labels["input_ids"]
    ]

    model_input["labels"] = label_ids
    return model_input

## Create HuggingFace Datasets

In [ ]:
from datasets import Dataset

# Redefine tokenize_example to fix the AttributeError
def tokenize_example(example):
    model_input = tokenizer(
        example["input_text"],
        max_length=64,
        padding="max_length",
        truncation=True
    )

    # Removed 'with tokenizer.as_target_tokenizer():' as it's deprecated
    labels = tokenizer(
        example["target_text"],
        max_length=8,
        padding="max_length",
        truncation=True
    )

    label_ids = [
        (token if token != tokenizer.pad_token_id else -100)
        for token in labels["input_ids"]
    ]

    model_input["labels"] = label_ids
    return model_input

train_dataset = Dataset.from_list(train_data)
dev_dataset   = Dataset.from_list(dev_data)
test_dataset  = Dataset.from_list(test_data)

tokenized_train = train_dataset.map(tokenize_example)
tokenized_dev   = dev_dataset.map(tokenize_example)
tokenized_test  = test_dataset.map(tokenize_example)

print("Tokenization done.")
print("Train:", len(tokenized_train), "Dev:", len(tokenized_dev), "Test:", len(tokenized_test))

## Normalize Labels & Compute Metrics

In [ ]:
import re
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

ALLOWED_LABELS = {"positive", "negative"}

def normalize_label(text):
    if text is None:
        return "negative"
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    text = text.split()[0] if text.split() else ""
    return text if text in ALLOWED_LABELS else "negative"

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # preds may be logits (3D) or token ids (2D)
    if isinstance(preds, tuple):
        preds = preds[0]
    if preds.ndim == 3:
        preds = preds.argmax(-1)

    # Replace -100 in labels with pad token id before decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    norm_preds  = [normalize_label(p) for p in decoded_preds]
    norm_labels = [normalize_label(l) for l in decoded_labels]

    acc = accuracy_score(norm_labels, norm_preds)
    f1  = f1_score(norm_labels, norm_preds, average="weighted", zero_division=0)

    print(f"\n--- Dev Metrics --- Accuracy: {acc:.4f} | F1 (weighted): {f1:.4f}")
    return {"accuracy": acc, "f1_weighted": f1}

print("compute_metrics defined.")

## Training Arguments

Key fixes:
- `eval_strategy='epoch'` so evaluation runs after each epoch
- `predict_with_generate=True` so the seq2seq model generates text for evaluation
- `load_best_model_at_end=True` to keep the best checkpoint
- Added `generation_max_length` to match label length

In [ ]:
from transformers import Seq2SeqTrainingArguments
import torch

training_args = Seq2SeqTrainingArguments(
    output_dir="./outputs",

    # ---- FIXED: enable evaluation ----
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    save_total_limit=2,

    # ---- FIXED: use generate for seq2seq evaluation ----
    predict_with_generate=True,
    generation_max_length=8,

    learning_rate=3e-4,          # higher LR helps small mT5 converge faster
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=50,
    fp16=False,
    report_to="none",
)
print("Training args configured.")

## Trainer

In [ ]:
from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq

# DataCollator handles dynamic padding properly
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_dev,        # FIXED: pass dev set
    compute_metrics=compute_metrics,   # FIXED: pass metrics function
    data_collator=data_collator,
)
print("Trainer ready.")

## Train

In [ ]:
train_result = trainer.train()
print("\n=== Training complete ===")
print(f"  Train loss:    {train_result.training_loss:.4f}")
print(f"  Train runtime: {train_result.metrics.get('train_runtime', 0):.1f}s")

## Evaluate on Dev Set

In [ ]:
dev_metrics = trainer.evaluate(eval_dataset=tokenized_dev)
print("\n=== Dev Set Results ===")
for k, v in dev_metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

## Evaluate on Test Set

In [ ]:
# Tokenize test set and run prediction
test_preds = trainer.predict(tokenized_test)

# Decode predictions
raw_preds = test_preds.predictions
if isinstance(raw_preds, tuple):
    raw_preds = raw_preds[0]
if raw_preds.ndim == 3:
    raw_preds = raw_preds.argmax(-1)

decoded_test_preds  = tokenizer.batch_decode(raw_preds, skip_special_tokens=True)
test_labels_arr     = np.where(test_preds.label_ids != -100,
                                test_preds.label_ids,
                                tokenizer.pad_token_id)
decoded_test_labels = tokenizer.batch_decode(test_labels_arr, skip_special_tokens=True)

norm_test_preds  = [normalize_label(p) for p in decoded_test_preds]
norm_test_labels = [normalize_label(l) for l in decoded_test_labels]

test_acc = accuracy_score(norm_test_labels, norm_test_preds)
test_f1  = f1_score(norm_test_labels, norm_test_preds, average="weighted", zero_division=0)

print("\n=== Test Set Results ===")
print(f"  Accuracy:    {test_acc:.4f}")
print(f"  F1 (weighted): {test_f1:.4f}")

# Show a sample of predictions
print("\n--- Sample Predictions (first 10) ---")
for i in range(min(10, len(norm_test_preds))):
    print(f"  Pred: {norm_test_preds[i]:<10}  Gold: {norm_test_labels[i]}")

In [ ]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

print("F1 (macro):", f1_score(norm_test_labels, norm_test_preds, average="macro", zero_division=0))
print("\nClassification report:\n", classification_report(norm_test_labels, norm_test_preds, digits=4, zero_division=0))

labels_order = ["positive", "negative"]
cm = confusion_matrix(norm_test_labels, norm_test_preds, labels=labels_order)

plt.figure()
plt.imshow(cm)
plt.xticks(range(len(labels_order)), labels_order, rotation=30)
plt.yticks(range(len(labels_order)), labels_order)
plt.title("Test Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
for i in range(len(labels_order)):
    for j in range(len(labels_order)):
        plt.text(j, i, str(cm[i][j]), ha="center", va="center")
plt.tight_layout()
plt.show()

## Save Model & Training History

In [ ]:
import os

# Save locally in Colab first, then copy to Drive
save_path = "/content/mt5_finetuned_updated"
os.makedirs(save_path, exist_ok=True)
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved locally to: {save_path}")

# Now copy to Drive
drive_path = "/content/drive/MyDrive/mt5_finetuned_updated"
os.system(f"cp -r {save_path} '{drive_path}'")
print(f"Copied to Drive: {drive_path}")

# Show training log
print("\n--- Training Log History ---")
for entry in trainer.state.log_history:
    relevant = {k: round(v, 4) if isinstance(v, float) else v
                for k, v in entry.items()
                if k in ("epoch", "loss", "eval_accuracy", "eval_f1_weighted",
                         "eval_loss", "train_loss")}
    if relevant:
        print(relevant)

In [ ]:
for i in range(10):
    print(f"Text: {test_data[i]['input_text']}")
    print(f"Pred: {norm_test_preds[i]}  |  Gold: {norm_test_labels[i]}")
    print()